# HR Assistant — LangGraph Walkthrough

This notebook walks through the HR Assistant demo built on LangGraph. It maps 1:1 onto the Assitio architecture from `Assitio_Architecture_Overview_v2.docx`, with all external dependencies mocked.

**Sections:**
1. Graph visualization
2. State schema
3. Normal scenario (with per-node state deltas)
4. Interrupt scenario
5. Multi-turn with checkpointer
6. Audit trail table

## 1. Graph visualization

In [ ]:
from hr_assistant.graph import build_graph
from hr_assistant.logging import configure_logging
configure_logging()
graph = build_graph()
from IPython.display import Image
Image(graph.get_graph().draw_mermaid_png())

## 2. State schema

In [ ]:
from hr_assistant.state import HRState
import typing
for field, annotation in typing.get_type_hints(HRState, include_extras=True).items():
    print(f'{field}: {annotation}')

## 3. Normal scenario with per-node deltas

In [ ]:
from hr_assistant.graph import default_input
cfg = {'configurable': {'thread_id': 'nb-normal'}}
state = default_input('What are the rules for parental leave?')
async for update in graph.astream(state, config=cfg):
    for node, delta in update.items():
        print(f'=== {node} ===')
        for k, v in delta.items():
            print(f'  {k}: {v}')

## 4. Interrupt scenario

In [ ]:
from langgraph.types import Command
cfg = {'configurable': {'thread_id': 'nb-interrupt'}}
state = default_input('Patient CPR 0102031234 parental leave?')
first = await graph.ainvoke(state, config=cfg)
print('Interrupt payload:', first.get('__interrupt__'))

In [ ]:
resumed = await graph.ainvoke(Command(resume={'approved': False}), config=cfg)
print('Final response:', resumed.get('response_text'))
print('Approval:', resumed.get('human_approval'))

## 5. Multi-turn with checkpointer

In [ ]:
cfg = {'configurable': {'thread_id': 'nb-multi'}}
await graph.ainvoke(default_input('Parental leave rules?'), config=cfg)
snap = await graph.aget_state(cfg)
print('Turn 1 routing path:', snap.values['routing_path'])

## 6. Audit trail table

In [ ]:
snap = await graph.aget_state({'configurable': {'thread_id': 'nb-normal'}})
from itertools import groupby
print('Step  | Node')
print('------+------------------------------')
for i, step in enumerate(snap.values.get('routing_path', []), 1):
    print(f'  {i:>2}  | {step}')